In [11]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/ECGPipes
Working Dir Base: /Users/peli/Projects/Repositories/ECGPipes


In [12]:
import yaml
import time
from nipype import Workflow, Node, MapNode, Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
# import
from src.preprocessing import *
from src.interfaces.initpreproc import InitialPreproc

loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


In [ ]:
def create_meg_preprocessing(
    basedir: str,
    workdir: str,
    output_dir: str,
    subject_list: list[str],
    crop_params: dict,
    filter_params: dict,
    gradcomp_params: dict,
):
    """
    MEG preprocessing pipeline using classic Nipype iterables pattern.
    
    Parallelization: infosource.iterables creates one workflow execution per subject.
    No MapNode needed—iterables handles subject-level iteration.
    """
        
    # Create workflow
    wf = Workflow(name="megpreproc")
    wf.base_dir = workdir
    
    # === SUBJECT ITERATION (classic Nipype pattern) ===
    infosource = Node(
        IdentityInterface(fields=['subject_id']),
        name="infosource"
    )
    infosource.iterables = [('subject_id', subject_list)]  # ← This creates parallel executions
    
    # === FILE SELECTION ===
    templates = {"meg": "{subject_id}/meg/{subject_id}_task-MMNHCS_run-0_meg.fif"}
    selectraw = Node(
        SelectFiles(templates, base_directory=basedir),
        name="selectfiles"
    )
    
    # === PROCESSING NODE (regular Node, NOT MapNode) ===
    # iterables handles the iteration, so no iterfield needed
    initial_preproc = Node(
        InitialPreproc(),
        name='initial_preproc'
    )
    
    # Set processing parameters
    initial_preproc.inputs.update({
        # Crop params
        "stim_channel": crop_params["stim_channel"],
        "min_buffer": crop_params["min_buffer"],
        "max_buffer": crop_params["max_buffer"],
        # Filter params
        "l_freq": filter_params["l_freq"],
        "h_freq": filter_params["h_freq"],
        # Gradcomp params
        "gradcomp_auto": gradcomp_params["auto"],
        "gradcomp_order": gradcomp_params["order"],
        # Output file (each subject gets their own workdir subdir)
        "out_file": "preproc_raw.fif",
    })
    
    # === OUTPUT ===
    datasink = Node(
        DataSink(
            base_directory=output_dir,
            container="preprocessed",
            parameterization=False  # Clean output paths
        ),
        name="datasink"
    )
    
    # === CONNECTIONS ===
    wf.connect([
        (infosource, selectraw, [("subject_id", "subject_id")]),
        (selectraw, initial_preproc, [("meg", "in_file")]),
        (initial_preproc, datasink, [("out_file", "megpreproc.@final")]),
    ])
    
    # Optional: log workflow structure (after creation, not during)
    logger.info(f"Created workflow with {len(subject_list)} subjects")
    logger.debug(f"Subject list: {subject_list}")
    
    return wf

In [14]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_ds006629.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

paths_config = config["paths"]
proc_config = config["processing"]
wf_config = config["workflow"]

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_meg_preprocessing(
    basedir=paths_config["basedir"],
    workdir=paths_config["workdir"],
    output_dir=paths_config["outputdir"],
    subject_list=paths_config["subjects"],  # ← Now actually used!
    crop_params=proc_config["crop"],
    filter_params=proc_config["filter"],
    gradcomp_params=proc_config["gradcomp"]
)

# Optional: visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

AttributeError: 'InitialPreprocInputSpec' object has no attribute 'update'

In [ ]:
# Write graph of type colored
wf.write_graph(graph2use='colored', dotfilename='./graph_colored.dot')

# Visualize graph
from IPython.display import Image
Image(filename="megpreproc/graph_colored.png")

OSError: No command "dot" found on host Elisiuss-MacBook-Pro.local. Please check that the corresponding package is installed.